# EDA — Exploring a New Dataset

## Notebook goal

In this notebook, we explore a new food delivery dataset before cleaning or modeling.

The goal is to understand:

- what the dataset contains,
- how the columns are structured,
- what types of values appear in the columns,
- where missing or inconsistent values exist,
- which problems need to be fixed during the cleaning phase.

We will not clean the data in this notebook.  
We will only inspect it and create a clear cleaning checklist.

## 1. Intro

Before training a regression model, we need to understand the data.

A raw dataset can contain many problems:

- inconsistent column names,
- missing values,
- numbers stored as text,
- categorical values written inconsistently,
- invalid dates or times,
- invalid coordinates,
- target values stored in the wrong format.

If we train a model before checking these issues, the model may fail, or it may learn from unreliable data.

In this notebook, the goal is to inspect the dataset and identify what needs to be cleaned later.

## 2. Import libraries

We start by importing the libraries needed for basic data exploration.

In [ ]:
from pathlib import Path

import pandas as pd

## 3. Define the data path

The path below assumes the following project structure:

```text
project/
├── data/
│   └── deliveries_dataset_raw.csv
└── notebooks/
    └── 01_eda_deliveries_dataset.ipynb
```

If your file has a different name or location, update the path.

In [ ]:
DATA_PATH = Path("../data/deliveries_dataset_raw.csv")

DATA_PATH

## 4. Load the dataset

We load the raw CSV file into a pandas DataFrame.

In [ ]:
df = pd.read_csv(DATA_PATH)

## 5. First look at the data

The first few rows give us an initial impression of the dataset.

In [ ]:
df.head()

It is also useful to inspect a few random rows, because the first rows are not always representative.

In [ ]:
df.sample(5, random_state=42)

## 6. Basic dataset structure

Now we check the basic structure of the dataset.

In [ ]:
df.shape

In [ ]:
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Next, we inspect the column names.

In [ ]:
df.columns.tolist()

The `info()` method shows column names, detected data types, and the number of non-missing values.

In [ ]:
df.info()

From this initial overview, we can already identify several issues that will need to be handled during data cleaning:

- column names should be standardized so they are easier to use in Python code,
- date and time columns were not detected as proper date/time types,
- missing values appear in many columns,
- the target column needs to be inspected more carefully before it can be used for regression.

These observations give us the first cleaning tasks that will be addressed later in the project.

## 7. Understanding the meaning of columns

Before cleaning the data, we need to understand what the columns represent.

We can roughly group the columns into:

- identifier columns,
- delivery person information,
- location information,
- date and time information,
- weather and traffic information,
- order and vehicle information,
- target column.

This helps us decide how each group of columns should be inspected.

Typical identifier columns are:

In [ ]:
identifier_columns = [
    "ID",
    "Delivery_person_ID",
]

df[identifier_columns].head()

Typical delivery person columns are:

In [ ]:
delivery_person_columns = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
]

df[delivery_person_columns].head()

Typical location columns are:

In [ ]:
location_columns = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
]

df[location_columns].head()

Typical date and time columns are:

In [ ]:
date_time_columns = [
    "Order_Date",
    "Time_Orderd",
    "Time_Order_picked",
]

df[date_time_columns].head()

Typical weather, traffic, order, and vehicle columns are:

In [ ]:
context_columns = [
    "Weather_conditions",
    "Road_traffic_density",
    "Type_of_order",
    "Type_of_vehicle",
    "Vehicle_condition",
    "multiple_deliveries",
    "Festival",
    "City",
]

df[context_columns].head()

The target column is:

In [ ]:
target_column = "Time_taken (min)"

df[[target_column]].head()

## 8. Target column inspection

The target column contains the value we want to predict.

For this project, the target is delivery time in minutes.

In [ ]:
df[target_column].head(10)

We check the detected data type.

In [ ]:
df[target_column].dtype

We also inspect a few unique values to understand how the target is written.

In [ ]:
df[target_column].unique()[:10]

The target column is already stored in a numeric format.

Since pandas detected this column as `int64`, the values are already integers and can be used as numeric target values for a regression model.

This also means that the column cannot contain text values such as `"(min) 24"` or other non-numeric strings in its current form. If such values existed, pandas would not have detected the column as `int64`.

Because of that, the target column does not require text extraction or conversion in the cleaning phase. However, we should still check whether it contains missing values or unusually small or large delivery times.

## 9. Missing values

First, we check real missing values detected by pandas.

In [ ]:
missing_values = df.isna().sum()

missing_values[missing_values > 0]

Raw data can also contain missing-like text values, such as `"NaN"`, `"null"`, `"None"`, or empty strings.

These values may look like regular text, but they usually mean that the value is missing.

In [ ]:
missing_like_values = ["NaN", "nan", "NULL", "null", "None", "none", "", " "]

for column in df.columns:
    if df[column].dtype == "object":
        count = df[column].astype(str).str.strip().isin(missing_like_values).sum()

        if count > 0:
            print(f"{column}: {count}")

A certain number of columns contain missing values.

Later, during the cleaning and preprocessing phases, we will decide how to handle them. In some cases, rows with missing values may be removed completely. In other cases, we may use an imputation strategy and fill missing values with appropriate replacement values.

The important thing in this notebook is to identify where missing values appear, so we can decide how to handle them in the next steps.

## 10. Numeric column inspection

Next, we inspect columns that should represent numeric values.

In [ ]:
numeric_columns = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
    "Vehicle_condition",
    "multiple_deliveries",
]

First, we check their detected data types.

In [ ]:
df[numeric_columns].dtypes

All selected numeric columns were detected as numeric types.

The only irregularity we can notice is that some columns use a `float` type even though their values appear to be whole numbers. This is most likely caused by missing values, because pandas often converts integer columns with missing values into floating-point columns.

Because of that, these columns do not require text-to-number conversion, but we will still need to decide how to handle missing values later in the cleaning and preprocessing phases.

## 11. Categorical column inspection

Now we inspect columns that contain categorical values.

In [ ]:
categorical_columns = [
    "Weather_conditions",
    "Road_traffic_density",
    "Type_of_order",
    "Type_of_vehicle",
    "Festival",
    "City",
]

First, we check how many unique values each categorical column contains.

In [ ]:
for column in categorical_columns:
    print(column)
    print(df[column].nunique(dropna=False))
    print("-" * 40)

Then we inspect the actual values and their frequencies.

In [ ]:
for column in categorical_columns:
    print(df[column].value_counts(dropna=False))
    print("-" * 40)

Based on this initial inspection, the categorical columns look consistent.

The expected categorical columns were correctly identified as categorical values, and there are no obvious cases where the same category appears in multiple different forms.

At this stage, we do not see major issues such as duplicated category labels, inconsistent capitalization, or unnecessary variations of the same value. These columns will still need to be encoded later, but they do not appear to require significant category cleaning.

## 12. Date and time column inspection

The dataset contains date and time columns.

We first check their detected data types.

In [ ]:
df[date_time_columns].dtypes

Then we inspect a few values.

In [ ]:
df[date_time_columns].head()

In [ ]:
df[date_time_columns].sample(10, random_state=42)

Next, we check whether the values follow the expected text patterns.

Expected date format:

```text
DD-MM-YYYY
```

Expected time formats:

```text
HH:MM
HH:MM:SS
```

Missing values are ignored in this check.

In [ ]:
date_pattern = r"\d{2}-\d{2}-\d{4}"
time_pattern = r"\d{2}:\d{2}(:\d{2})?"

invalid_dates = (
    df["Order_Date"].notna()
    & ~df["Order_Date"].astype(str).str.strip().str.fullmatch(date_pattern)
)

invalid_order_times = (
    df["Time_Orderd"].notna()
    & ~df["Time_Orderd"].astype(str).str.strip().str.fullmatch(time_pattern)
)

invalid_pickup_times = (
    df["Time_Order_picked"].notna()
    & ~df["Time_Order_picked"].astype(str).str.strip().str.fullmatch(time_pattern)
)

print("Invalid dates:", invalid_dates.sum())
print("Invalid order times:", invalid_order_times.sum())
print("Invalid pickup times:", invalid_pickup_times.sum())

Let's see some of the invalid values:

In [ ]:
print("Invalid date examples:")
print(df.loc[invalid_dates, "Order_Date"].drop_duplicates().head(10).to_list())

print("Invalid order time examples:")
print(df.loc[invalid_order_times, "Time_Orderd"].drop_duplicates().head(10).to_list())

print("Invalid pickup time examples:")
print(df.loc[invalid_pickup_times, "Time_Order_picked"].drop_duplicates().head(10).to_list())

The date and time columns were not detected as specific datetime data types.

They are still stored as regular text/object values, which means we will need to parse and convert them later during the cleaning phase.

The format check also shows that a certain number of rows contain invalid time values. These values will need special attention before we can safely create proper datetime columns or use time-based information in later steps.

## 13. Coordinate inspection

The dataset contains restaurant coordinates and delivery location coordinates.

In [ ]:
coordinate_columns = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
]

First, we check the detected data types.

In [ ]:
df[coordinate_columns].dtypes

Latitude should be between `-90` and `90`.  
Longitude should be between `-180` and `180`.

In [ ]:
invalid_coordinates = (
    ~df["Restaurant_latitude"].between(-90, 90)
    | ~df["Delivery_location_latitude"].between(-90, 90)
    | ~df["Restaurant_longitude"].between(-180, 180)
    | ~df["Delivery_location_longitude"].between(-180, 180)
)

print("Invalid coordinate rows:", invalid_coordinates.sum())

We also check coordinates that are equal to zero.

In this project, zero coordinates are suspicious because they are unlikely to represent real restaurant or delivery locations.

In [ ]:
zero_coordinates = (
    (df["Restaurant_latitude"] == 0)
    | (df["Restaurant_longitude"] == 0)
    | (df["Delivery_location_latitude"] == 0)
    | (df["Delivery_location_longitude"] == 0)
)

print("Rows with zero coordinates:", zero_coordinates.sum())

Some values may not be exactly zero, but still suspiciously close to zero.

For this dataset, near-zero coordinates are also unlikely to represent valid locations.

In [ ]:
near_zero_coordinates = (
    df["Restaurant_latitude"].between(-1, 1)
    | df["Restaurant_longitude"].between(-1, 1)
    | df["Delivery_location_latitude"].between(-1, 1)
    | df["Delivery_location_longitude"].between(-1, 1)
)

print("Rows with near-zero coordinates:", near_zero_coordinates.sum())

We can inspect a few problematic coordinate rows.

In [ ]:
problematic_coordinates = (
    invalid_coordinates
    | zero_coordinates
    | near_zero_coordinates
)

df.loc[problematic_coordinates, coordinate_columns].head()

The coordinate columns were detected with appropriate numeric data types.

However, we also detected coordinates that are equal to zero or very close to zero. For this project, such values are considered invalid because they are unlikely to represent real restaurant or delivery locations.

These problematic coordinate values will need to be handled during the cleaning phase.

## 14. Duplicate rows

Duplicate rows can distort analysis and model training.

We check whether complete duplicate rows exist.

In [ ]:
duplicate_rows = df.duplicated()

print("Duplicate rows:", duplicate_rows.sum())

In [ ]:
df.loc[duplicate_rows].head()

The check shows that there are no complete duplicate rows in the dataset.

Because of that, we do not need to remove duplicate rows during the cleaning phase.

## 15. Final EDA Findings and Cleaning Plan

Based on the exploratory analysis, we identified the following tasks that need to be handled before model training:

- Standardize column names so they are easier and safer to use in Python code.
- Check the target column for missing values and unusually small or large delivery times.
- Handle missing values in several columns.  
- Review numeric columns that use a `float` type even though they contain whole-number values.  
- Convert date and time columns into proper datetime formats, because they were not detected as specific datetime data types.
- Handle invalid time values detected in some rows before creating proper datetime columns.
- Handle invalid coordinate values, especially coordinates that are equal to zero or very close to zero.

The result of the cleaning phase should be a cleaner and more reliable version of the dataset, ready for the next stages of the machine learning workflow.